In [1]:
import random
import time

import tensorflow as tf
from tensorflow.keras import Sequential

from tensorflow.keras.layers import (
    Input,
    Conv1D,
    BatchNormalization,
    Activation,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout,
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix,  precision_score, recall_score, f1_score, accuracy_score,roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, Normalizer, OrdinalEncoder, MinMaxScaler,  LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import SelectKBest, chi2, SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

import numpy as np
import pandas as pd


In [5]:
data_1 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv")
data_2 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv")
data_3 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv")
data_4 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv")
data_5 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv")
data_6 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv")
data_7 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Friday-23-02-2018_TrafficForML_CICFlowMeter.csv")
data_8 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv")
data_9 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv")
data_10 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv")

/tmp/ipykernel_4983/3643216608.py:3: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  data_3 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv")
/tmp/ipykernel_4983/3643216608.py:6: DtypeWarning: Columns (0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  data_6 = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv")
/tmp/ipy

In [6]:
extra_cols = ['Flow ID', 'Src IP', 'Src Port', 'Dst IP']
data_9 = data_9.drop(columns=[c for c in extra_cols if c in data_9.columns])

In [7]:
data_list = [data_1, data_2, data_3, data_4, data_5, data_6, data_7, data_8, data_9, data_10]
dataframe = pd.concat(data_list)
rows, cols = dataframe.shape

print('New dimension:')
print(f'Number of rows: {rows}')
print(f'Number of columns: {cols}')
print(f'Total cells: {rows * cols}')

New dimension:
Number of rows: 16233002
Number of columns: 80
Total cells: 1298640160


In [8]:
for d in data_list: del d

# **A cloud intrusion detection systems based on dnn using backpropagation and pso on the cse-cic ids2018 dataset**

In [9]:
random.seed(42)
np.random.seed(42)
tf.keras.utils.set_random_seed(42)

In [10]:
import pandas as pd

dataframe.columns = dataframe.columns.astype(str).str.strip()

column_aliases = {
    "Destination Port": "Dst Port",
    "Flow Duration": "Flow Duration",
    "Fwd Packet Length Max": "Fwd Pkt Len Max",
    "Fwd Packet Length Mean": "Fwd Pkt Len Mean",
    "Total Length of Fwd Packets": "TotLen Fwd Pkts",
    "Total Fwd Packets": "Tot Fwd Pkts",
    "Fwd IAT Total": "Fwd IAT Tot",
    "Fwd Header Length": "Fwd Header Len",
    "Fwd Packets/s": "Fwd Pkts/s",
    "Bwd Packets/s": "Bwd Pkts/s",
    "Flow Packets/s": "Flow Pkts/s",
    "Packet Length Max": "Pkt Len Max",
    "Fwd Segment Size Avg": "Fwd Seg Size Avg",
    "Fwd Seg Size Min": "Fwd Seg Size Min",
    "Init_Win_bytes_forward": "Init Fwd Win Byts",
    "Init_Win_bytes_backward": "Init Bwd Win Byts",
    "Subflow Fwd Bytes": "Subflow Fwd Byts",
    "Subflow Fwd Packets": "Subflow Fwd Pkts",
    "Source IP": "Src IP",
    "Destination IP": "Dst IP",
    "Source Port": "Src Port",
}

dataframe = dataframe.rename(
    columns={
        existing_column: column_aliases[existing_column]
        for existing_column in dataframe.columns
        if existing_column in column_aliases
    }
)

print("Columns after normalization:", len(dataframe.columns))

Columns after normalization: 80


In [11]:
print("Shape before cleaning:", dataframe.shape)

dataframe = dataframe.replace([np.inf, -np.inf], np.nan)
dataframe = dataframe.dropna(axis=0).drop_duplicates().reset_index(drop=True)

columns_to_drop = ["Timestamp", "Flow ID", "Src IP", "Src Port", "Dst IP"]
dataframe = dataframe.drop(
    columns=[column for column in columns_to_drop if column in dataframe.columns],
    errors="ignore",
)

label_column = "Label"
dataframe[label_column] = dataframe[label_column].astype(str).str.strip()
dataframe["Binary Label"] = (
    dataframe[label_column].str.casefold() != "benign"
).astype("int8")

print("Shape after cleaning:", dataframe.shape)
print(dataframe["Binary Label"].value_counts())
print(dataframe["Binary Label"].value_counts(normalize=True))

Shape before cleaning: (16233002, 80)
Shape after cleaning: (15708545, 80)
Binary Label
0    13353852
1     2354693
Name: count, dtype: int64
Binary Label
0    0.850101
1    0.149899
Name: proportion, dtype: float64


In [14]:
for col in dataframe.columns:
    if col not in ['Label', 'Binary Label']:
        # Attempt to convert to numeric, coercing errors
        dataframe[col] = pd.to_numeric(dataframe[col], errors='coerce')

# Drop rows that resulted in NaN after numeric conversion
dataframe = dataframe.dropna().reset_index(drop=True)

print("Shape after numeric type conversion and NaN removal:", dataframe.shape)
print(dataframe.info())

Shape after numeric type conversion and NaN removal: (15708544, 80)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15708544 entries, 0 to 15708543
Data columns (total 80 columns):
 #   Column             Dtype  
---  ------             -----  
 0   Dst Port           float64
 1   Protocol           float64
 2   Flow Duration      float64
 3   Tot Fwd Pkts       float64
 4   Tot Bwd Pkts       float64
 5   TotLen Fwd Pkts    float64
 6   TotLen Bwd Pkts    float64
 7   Fwd Pkt Len Max    float64
 8   Fwd Pkt Len Min    float64
 9   Fwd Pkt Len Mean   float64
 10  Fwd Pkt Len Std    float64
 11  Bwd Pkt Len Max    float64
 12  Bwd Pkt Len Min    float64
 13  Bwd Pkt Len Mean   float64
 14  Bwd Pkt Len Std    float64
 15  Flow Byts/s        float64
 16  Flow Pkts/s        float64
 17  Flow IAT Mean      float64
 18  Flow IAT Std       float64
 19  Flow IAT Max       float64
 20  Flow IAT Min       float64
 21  Fwd IAT Tot        float64
 22  Fwd IAT Mean       float64
 23  Fwd IAT Std 

In [15]:
dataframe.to_parquet("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/cicids2018_preprocessed.parquet",index=False)

In [16]:
selected_features = [
    "Dst Port",
    "Fwd Seg Size Min",
    "Init Fwd Win Byts",
    "Fwd Pkt Len Max",
    "TotLen Fwd Pkts",
    "Subflow Fwd Byts",
    "Fwd Pkt Len Mean",
    "Fwd IAT Tot",
    "Flow Duration",
    "Fwd IAT Mean",
    "Fwd IAT Max",
    "Flow IAT Max",
    "Fwd Seg Size Avg",
    "Flow IAT Min",
    "Fwd Pkts/s",
    "Flow IAT Mean",
    "Fwd IAT Min",
    "Flow Pkts/s",
    "Fwd Header Len",
    "Init Bwd Win Byts",
    "Bwd Pkts/s",
    "Pkt Len Max",
    "Tot Fwd Pkts",
    "Subflow Fwd Pkts",
]





X = dataframe[selected_features].apply(pd.to_numeric, errors="coerce")
y = dataframe["Binary Label"].astype("int8")

valid_rows = X.notna().all(axis=1)
X = X.loc[valid_rows].reset_index(drop=True)
y = y.loc[valid_rows].reset_index(drop=True)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class counts:")
print(y.value_counts())

X shape: (15708544, 24)
y shape: (15708544,)
Class counts:
Binary Label
0    13353852
1     2354692
Name: count, dtype: int64


In [17]:
X_train_raw, X_test_raw, y_train_integer, y_test_integer = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
    shuffle=True,
)

print("Training rows:", X_train_raw.shape[0])
print("Testing rows:", X_test_raw.shape[0])
print("Training distribution:")
print(y_train_integer.value_counts(normalize=True))
print("Testing distribution:")
print(y_test_integer.value_counts(normalize=True))

Training rows: 10995980
Testing rows: 4712564
Training distribution:
Binary Label
0    0.850101
1    0.149899
Name: proportion, dtype: float64
Testing distribution:
Binary Label
0    0.850101
1    0.149899
Name: proportion, dtype: float64


In [19]:
scaler = StandardScaler()

X_train_raw = X_train_raw.replace([np.inf, -np.inf], np.nan)
X_test_raw = X_test_raw.replace([np.inf, -np.inf], np.nan)


valid_train_indices = X_train_raw.dropna().index
X_train_raw = X_train_raw.loc[valid_train_indices].reset_index(drop=True)
y_train_integer = y_train_integer.loc[valid_train_indices].reset_index(drop=True)


valid_test_indices = X_test_raw.dropna().index
X_test_raw = X_test_raw.loc[valid_test_indices].reset_index(drop=True)
y_test_integer = y_test_integer.loc[valid_test_indices].reset_index(drop=True)

X_train = scaler.fit_transform(X_train_raw).astype("float32")
X_test = scaler.transform(X_test_raw).astype("float32")

try:
    label_encoder = OneHotEncoder(
        categories=[[0, 1]],
        sparse_output=False,
        dtype=np.float32,
        handle_unknown="ignore",
    )
except TypeError:
    label_encoder = OneHotEncoder(
        categories=[[0, 1]],
        sparse=False,
        dtype=np.float32,
        handle_unknown="ignore",
    )

y_train = label_encoder.fit_transform(
    y_train_integer.to_numpy().reshape(-1, 1)
)
y_test = label_encoder.transform(
    y_test_integer.to_numpy().reshape(-1, 1)
)

print("X_train:", X_train.shape, X_train.dtype)
print("X_test:", X_test.shape, X_test.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("y_test:", y_test.shape, y_test.dtype)

X_train: (10995270, 24) float32
X_test: (4712250, 24) float32
y_train: (10995270, 2) float32
y_test: (4712250, 2) float32


In [20]:
learning_rate_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.001,
    decay_steps=10000,
    decay_rate=0.96,
    staircase=True,
)

model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(24,)),
        tf.keras.layers.Dense(15, activation="softsign"),
        tf.keras.layers.Dense(11, activation="relu"),
        tf.keras.layers.Dense(2, activation="softmax"),
    ]
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=learning_rate_schedule
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[tf.keras.metrics.CategoricalAccuracy(name="accuracy")],
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 15)             │           375 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 11)             │           176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            24 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 575 (2.25 KB)

 Trainable params: 575 (2.25 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
history = model.fit(
    X_train,
    y_train,
    batch_size=2048,
    epochs=100,
    validation_split=0.10,
    shuffle=True,
    verbose=1,
)

Epoch 1/100
4832/4832 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - accuracy: 0.9669 - loss: 0.1136 - val_accuracy: 0.9859 - val_loss: 0.0634
Epoch 2/100
4832/4832 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9865 - loss: 0.0605 - val_accuracy: 0.9872 - val_loss: 0.0580
Epoch 3/100
4832/4832 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9874 - loss: 0.0570 - val_accuracy: 0.9877 - val_loss: 0.0561
Epoch 4/100
4832/4832 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9877 - loss: 0.0556 - val_accuracy: 0.9879 - val_loss: 0.0550
Epoch 5/100
4832/4832 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9879 - loss: 0.0548 - val_accuracy: 0.9880 - val_loss: 0.0543
Epoch 6/100
4832/4832 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9880 - loss: 0.0543 - val_accuracy: 0.9881 - val_loss: 0.0539
Epoch 7/100
4832/4832 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9880 - loss: 0.0540 - val_accuracy: 0.9882 - val_loss: 0.0537
Epoch 8/100
4832/4832 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9880 -

In [23]:
test_probabilities = model.predict(
    X_test,
    batch_size=2048,
    verbose=1,
)

y_pred = np.argmax(test_probabilities, axis=1)
y_test = y_test_integer.to_numpy()

2301/2301 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step


In [24]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=[0, 1],
)
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test, y_pred)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[4000887    4981]
 [  48157  658225]]

Attack-class metrics
--------------------
Accuracy : 0.9887 (98.87%)
Detection rate : 0.9318 (93.18%)
Precision      : 0.9925 (99.25%)
FPR            : 0.0012 (0.12%)
F1 score       : 0.9612 (96.12%)


In [25]:
model.save("/content/drive/MyDrive/Models_Table_7/MLP_Alzughabi.keras")

# **Network Anomaly Intrusion Detection Based on Deep Learning Approach**

In [3]:
dataframe = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/cicids2018_preprocessed.parquet")

In [4]:
random.seed(42)
np.random.seed(42)
tf.keras.utils.set_random_seed(42)

In [5]:
dataframe.columns = dataframe.columns.astype(str).str.strip()

column_aliases = {
    "Source IP": "Src IP",
    "Source Port": "Src Port",
    "Destination IP": "Dst IP",
    "Destination Port": "Dst Port",
    "Flow Bytes/s": "Flow Byts/s",
    "Flow Packets/s": "Flow Pkts/s",
    "Fwd Avg Bytes/Bulk": "Fwd Byts/b Avg",
    "Fwd Avg Packets/Bulk": "Fwd Pkts/b Avg",
    "Fwd Avg Bulk Rate": "Fwd Blk Rate Avg",
    "Bwd Avg Bytes/Bulk": "Bwd Byts/b Avg",
    "Bwd Avg Packets/Bulk": "Bwd Pkts/b Avg",
    "Bwd Avg Bulk Rate": "Bwd Blk Rate Avg",
}

dataframe = dataframe.rename(
    columns={
        column: column_aliases[column]
        for column in dataframe.columns
        if column in column_aliases
    }
)

print(dataframe.columns.tolist())

['Dst Port', 'Protocol', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt', 'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Down/Up Ratio', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'B

In [6]:
label_column = "Label"
dataframe[label_column] = dataframe[label_column].astype(str).str.strip()

invalid_label_values = {
    "",
    "nan",
    "none",
    "label",
}

valid_label_mask = ~dataframe[label_column].str.casefold().isin(
    invalid_label_values
)

dataframe = dataframe.loc[valid_label_mask].reset_index(drop=True)

identifier_columns = [
    "Timestamp",
    "Flow ID",
    "Src IP",
    "Src Port",
    "Dst IP",
    "Dst Port",
]

all_zero_columns = [
    "Bwd PSH Flags",
    "Bwd URG Flags",
    "Fwd Byts/b Avg",
    "Fwd Pkts/b Avg",
    "Fwd Blk Rate Avg",
    "Bwd Byts/b Avg",
    "Bwd Pkts/b Avg",
    "Bwd Blk Rate Avg",
]

dataframe = dataframe.drop(
    columns=[
        column
        for column in identifier_columns + all_zero_columns
        if column in dataframe.columns
    ],
    errors="ignore",
)

print("Shape after label and feature removal:", dataframe.shape)

Shape after label and feature removal: (15708544, 71)


In [7]:
for column in ["Flow Byts/s", "Flow Pkts/s"]:
    if column in dataframe.columns:
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        )
        dataframe[column] = dataframe[column].replace(
            [np.inf, -np.inf],
            np.nan,
        )

        column_mode = dataframe[column].mode(dropna=True)

        if column_mode.empty:
            raise ValueError(f"No finite mode available for {column}")

        dataframe[column] = dataframe[column].fillna(
            column_mode.iloc[0]
        )

print(
    dataframe[
        [
            column
            for column in ["Flow Byts/s", "Flow Pkts/s"]
            if column in dataframe.columns
        ]
    ].isna().sum()
)

Flow Byts/s    0
Flow Pkts/s    0
dtype: int64


In [8]:
import gc

predictor_columns = [
    column
    for column in dataframe.columns
    if column != "Label"
]

for column in predictor_columns:
    dataframe[column] = pd.to_numeric(
        dataframe[column],
        errors="coerce",
    )

dataframe[predictor_columns] = dataframe[
    predictor_columns
].replace([np.inf, -np.inf], np.nan)

remaining_missing_rows = dataframe[
    predictor_columns
].isna().any(axis=1)

print(
    "Rows with other unresolved missing values:",
    int(remaining_missing_rows.sum()),
)

dataframe = dataframe.loc[
    ~remaining_missing_rows
].reset_index(drop=True)

rows_before_duplicates = dataframe.shape[0]
dataframe = dataframe.drop_duplicates().reset_index(drop=True)
rows_after_duplicates = dataframe.shape[0]

print("Removed duplicates:", rows_before_duplicates - rows_after_duplicates)
print("Processed shape:", dataframe.shape)

gc.collect()

Rows with other unresolved missing values: 0
Removed duplicates: 4889383
Processed shape: (10819161, 71)


0

In [9]:
dataframe["Binary Label"] = (
    dataframe["Label"].astype(str).str.strip().str.casefold()
    != "benign"
).astype("int8")

predictor_columns = [
    column
    for column in dataframe.columns
    if column not in ["Label", "Binary Label"]
]

constant_columns = [
    column
    for column in predictor_columns
    if dataframe[column].nunique(dropna=False) <= 1
]

if constant_columns:
    print("Additional constant columns found:", constant_columns)

predictor_columns = [
    column
    for column in predictor_columns
    if column not in constant_columns
]

if len(predictor_columns) < 69:
    raise ValueError(
        f"Only {len(predictor_columns)} predictors remain; paper expects 69."
    )

if len(predictor_columns) > 69:
    print(
        f"{len(predictor_columns)} predictors remain. "
        "Keeping the first 69 in dataset order to match the paper."
    )
    predictor_columns = predictor_columns[:69]

X_raw = dataframe[predictor_columns].copy()
y = dataframe["Binary Label"].copy()

print("X shape:", X_raw.shape)
print("Class distribution:")
print(y.value_counts())

X shape: (10819161, 69)
Class distribution:
Binary Label
0    9493294
1    1325867
Name: count, dtype: int64


In [10]:
X_train_validation_raw, X_test_raw, y_train_validation, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
    shuffle=True,
)

X_train_raw, X_validation_raw, y_train, y_validation = train_test_split(
    X_train_validation_raw,
    y_train_validation,
    test_size=0.20,
    random_state=42,
    stratify=y_train_validation,
    shuffle=True,
)

print("Train:", X_train_raw.shape, y_train.value_counts().to_dict())
print("Validation:", X_validation_raw.shape, y_validation.value_counts().to_dict())
print("Test:", X_test_raw.shape, y_test.value_counts().to_dict())

Train: (6924262, 69) {0: 6075707, 1: 848555}
Validation: (1731066, 69) {0: 1518927, 1: 212139}
Test: (2163833, 69) {0: 1898660, 1: 265173}


In [11]:
standard_scaler = StandardScaler()

X_train_standardized = standard_scaler.fit_transform(
    X_train_raw
).astype("float32")

X_validation_standardized = standard_scaler.transform(
    X_validation_raw
).astype("float32")

X_test_standardized = standard_scaler.transform(
    X_test_raw
).astype("float32")

training_minimums = X_train_standardized.min(
    axis=0,
    keepdims=True,
)

nonnegative_shifts = np.where(
    training_minimums < 0,
    -training_minimums,
    0.0,
).astype("float32")

X_train = np.log1p(
    np.maximum(
        X_train_standardized + nonnegative_shifts,
        0.0,
    )
).astype("float32")

X_validation = np.log1p(
    np.maximum(
        X_validation_standardized + nonnegative_shifts,
        0.0,
    )
).astype("float32")

X_test = np.log1p(
    np.maximum(
        X_test_standardized + nonnegative_shifts,
        0.0,
    )
).astype("float32")

y_train_array = y_train.to_numpy().astype("float32")
y_validation_array = y_validation.to_numpy().astype("float32")
y_test_array = y_test.to_numpy().astype("int8")

print("Train shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
print("Test shape:", X_test.shape)
print("Train range:", X_train.min(), X_train.max())

Train shape: (6924262, 69)
Validation shape: (1731066, 69)
Test shape: (2163833, 69)
Train range: 0.0 7.866797


In [13]:
cnn_model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(69, 1)),
        tf.keras.layers.Conv1D(
            128,
            kernel_size=2,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(
            128,
            kernel_size=2,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(
            256,
            kernel_size=2,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.Conv1D(
            256,
            kernel_size=2,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.MaxPooling1D(pool_size=2),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ],
    name="CNN_4_768_binary",
)

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")],
)

cnn_model.summary()

Model: "CNN_4_768_binary"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 69, 128)        │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 34, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 34, 128)        │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 17, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 17, 256)        │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 8, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 8, 256)         │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 4, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 4, 256)         │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4, 256)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,025 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 232,449 (908.00 KB)

 Trainable params: 231,937 (906.00 KB)

 Non-trainable params: 512 (2.00 KB)

In [14]:
cnn_history = cnn_model.fit(
    X_train[..., np.newaxis],
    y_train_array,
    validation_data=(
        X_validation[..., np.newaxis],
        y_validation_array,
    ),
    batch_size=2048,
    epochs=50,
    shuffle=True,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
        )
    ],
    verbose=1,
)

Epoch 1/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 40s 9ms/step - accuracy: 0.9791 - loss: 0.0794 - val_accuracy: 0.9852 - val_loss: 0.0610
Epoch 2/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - accuracy: 0.9858 - loss: 0.0614 - val_accuracy: 0.9850 - val_loss: 0.0646
Epoch 3/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - accuracy: 0.9876 - loss: 0.0569 - val_accuracy: 0.9885 - val_loss: 0.0545
Epoch 4/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - accuracy: 0.9870 - loss: 0.0575 - val_accuracy: 0.9853 - val_loss: 0.0583
Epoch 5/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - accuracy: 0.9872 - loss: 0.0566 - val_accuracy: 0.9884 - val_loss: 0.0535
Epoch 6/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - accuracy: 0.9875 - loss: 0.0562 - val_accuracy: 0.9887 - val_loss: 0.0529
Epoch 7/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - accuracy: 0.9883 - loss: 0.0540 - val_accuracy: 0.9840 - val_loss: 0.0582
Epoch 8/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - accuracy: 0.9878 - loss: 0

In [18]:
cnn_model.save("/content/drive/MyDrive/Models_Table_7/CNN_Wang.keras")

In [15]:
rnn_input = tf.keras.layers.Input(shape=(1, 69))

rnn_x = tf.keras.layers.SimpleRNN(
    128,
    return_sequences=True,
)(rnn_input)
rnn_x = tf.keras.layers.BatchNormalization()(rnn_x)
rnn_x = tf.keras.layers.Dropout(0.2)(rnn_x)

rnn_x = tf.keras.layers.SimpleRNN(
    128,
    return_sequences=True,
)(rnn_x)
rnn_x = tf.keras.layers.BatchNormalization()(rnn_x)
rnn_x = tf.keras.layers.Dropout(0.2)(rnn_x)

rnn_x = tf.keras.layers.SimpleRNN(
    128,
    return_sequences=True,
)(rnn_x)
rnn_x = tf.keras.layers.BatchNormalization()(rnn_x)
rnn_x = tf.keras.layers.Dropout(0.2)(rnn_x)

rnn_x = tf.keras.layers.SimpleRNN(
    128,
    return_sequences=True,
)(rnn_x)
rnn_x = tf.keras.layers.BatchNormalization()(rnn_x)
rnn_x = tf.keras.layers.Dropout(0.2)(rnn_x)

rnn_x = tf.keras.layers.SimpleRNN(
    256,
    return_sequences=False,
)(rnn_x)
rnn_x = tf.keras.layers.BatchNormalization()(rnn_x)
rnn_x = tf.keras.layers.Dropout(0.2)(rnn_x)

rnn_output = tf.keras.layers.Dense(
    1,
    activation="sigmoid",
)(rnn_x)

rnn_model = tf.keras.Model(
    rnn_input,
    rnn_output,
    name="RNN_5_768_binary",
)

rnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")],
)

rnn_model.summary()

Model: "RNN_5_768_binary"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 1, 69)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 1, 128)         │        25,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 1, 128)         │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 1, 128)         │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 1, 128)         │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 256)            │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,921 (882.50 KB)

 Trainable params: 224,385 (876.50 KB)

 Non-trainable params: 1,536 (6.00 KB)

In [16]:
rnn_history = rnn_model.fit(
    X_train[:, np.newaxis, :],
    y_train_array,
    validation_data=(
        X_validation[:, np.newaxis, :],
        y_validation_array,
    ),
    batch_size=2048,
    epochs=50,
    shuffle=True,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
        )
    ],
    verbose=1,
)

Epoch 1/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 43s 8ms/step - accuracy: 0.9809 - loss: 0.0782 - val_accuracy: 0.8506 - val_loss: 0.6199
Epoch 2/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9819 - loss: 0.0740 - val_accuracy: 0.9239 - val_loss: 0.2435
Epoch 3/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9775 - loss: 0.0873 - val_accuracy: 0.8345 - val_loss: 0.5180
Epoch 4/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9701 - loss: 0.1053 - val_accuracy: 0.9797 - val_loss: 0.0964
Epoch 5/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9489 - loss: 0.1469 - val_accuracy: 0.9728 - val_loss: 0.1162
Epoch 6/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9318 - loss: 0.1760 - val_accuracy: 0.9765 - val_loss: 0.0987
Epoch 7/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9036 - loss: 0.2452 - val_accuracy: 0.8775 - val_loss: 0.2060
Epoch 8/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.8783 - loss: 0

In [17]:
rnn_model.save("/content/drive/MyDrive/Models_Table_7/RNN_Wang.keras")

In [12]:
lstm_input = tf.keras.layers.Input(shape=(1, 69))

lstm_x = tf.keras.layers.LSTM(
    128,
    return_sequences=True,
)(lstm_input)
lstm_x = tf.keras.layers.BatchNormalization()(lstm_x)
lstm_x = tf.keras.layers.Dropout(0.2)(lstm_x)

lstm_x = tf.keras.layers.LSTM(
    128,
    return_sequences=True,
)(lstm_x)
lstm_x = tf.keras.layers.BatchNormalization()(lstm_x)
lstm_x = tf.keras.layers.Dropout(0.2)(lstm_x)

lstm_x = tf.keras.layers.LSTM(
    128,
    return_sequences=True,
)(lstm_x)
lstm_x = tf.keras.layers.BatchNormalization()(lstm_x)
lstm_x = tf.keras.layers.Dropout(0.2)(lstm_x)

lstm_x = tf.keras.layers.LSTM(
    128,
    return_sequences=True,
)(lstm_x)
lstm_x = tf.keras.layers.BatchNormalization()(lstm_x)
lstm_x = tf.keras.layers.Dropout(0.2)(lstm_x)

lstm_x = tf.keras.layers.LSTM(
    256,
    return_sequences=False,
)(lstm_x)
lstm_x = tf.keras.layers.BatchNormalization()(lstm_x)
lstm_x = tf.keras.layers.Dropout(0.2)(lstm_x)

lstm_output = tf.keras.layers.Dense(
    1,
    activation="sigmoid",
)(lstm_x)

lstm_model = tf.keras.Model(
    lstm_input,
    lstm_output,
    name="LSTM_5_768_binary",
)

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[tf.keras.metrics.BinaryAccuracy(name="accuracy")],
)

lstm_model.summary()

Model: "LSTM_5_768_binary"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1, 69)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 1, 128)         │       101,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 1, 128)         │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 1, 128)         │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 1, 128)         │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 1, 128)         │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 893,697 (3.41 MB)

 Trainable params: 892,161 (3.40 MB)

 Non-trainable params: 1,536 (6.00 KB)

In [13]:

lstm_history = lstm_model.fit(
    X_train[:, np.newaxis, :],
    y_train_array,
    validation_data=(
        X_validation[:, np.newaxis, :],
        y_validation_array,
    ),
    batch_size=2048,
    epochs=50,
    shuffle=True,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
        )
    ],
    verbose=1,
)

Epoch 1/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 63s 16ms/step - accuracy: 0.9816 - loss: 0.0755 - val_accuracy: 0.9620 - val_loss: 0.1601
Epoch 2/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 54s 16ms/step - accuracy: 0.9828 - loss: 0.0687 - val_accuracy: 0.8574 - val_loss: 0.4149
Epoch 3/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 54s 16ms/step - accuracy: 0.9800 - loss: 0.0784 - val_accuracy: 0.9713 - val_loss: 0.0960
Epoch 4/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 54s 16ms/step - accuracy: 0.9633 - loss: 0.1221 - val_accuracy: 0.9462 - val_loss: 0.1271
Epoch 5/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 53s 16ms/step - accuracy: 0.9220 - loss: 0.2223 - val_accuracy: 0.9044 - val_loss: 0.3044
Epoch 6/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 54s 16ms/step - accuracy: 0.9152 - loss: 0.2398 - val_accuracy: 0.9392 - val_loss: 0.2160
Epoch 7/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 53s 16ms/step - accuracy: 0.9095 - loss: 0.2618 - val_accuracy: 0.8775 - val_loss: 0.3265
Epoch 8/50
3381/3381 ━━━━━━━━━━━━━━━━━━━━ 54s 16ms/step - accuracy: 0.8775 -

In [14]:
lstm_model.save("/content/drive/MyDrive/Models_Table_7/LSTM_Wang.keras")

In [15]:
current_test_input = X_test[:, np.newaxis, :]
current_probabilities = lstm_model.predict(
        current_test_input,
        batch_size=2048,
        verbose=1,
    ).reshape(-1)

y_pred = (
    current_probabilities >= 0.5
).astype("int8")

cm = confusion_matrix(
    y_test_array,
    y_pred,
    labels=[0, 1],
)



1057/1057 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step


In [16]:
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test_array, y_pred)

recall = recall_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[1889694    8966]
 [  53097  212076]]

Attack-class metrics
--------------------
Accuracy : 0.9713 (97.13%)
Detection rate : 0.7998 (79.98%)
Precision      : 0.9594 (95.94%)
FPR            : 0.0047 (0.47%)
F1 score       : 0.8724 (87.24%)


In [17]:
cnn_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/Models_Table_7/CNN_Wang.keras"
)


current_test_input = X_test[..., np.newaxis]
current_probabilities = cnn_model.predict(
        current_test_input,
        batch_size=2048,
        verbose=1,
    ).reshape(-1)

y_pred = (
    current_probabilities >= 0.5
).astype("int8")

cm = confusion_matrix(
    y_test_array,
    y_pred,
    labels=[0, 1],
)

1057/1057 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step


In [18]:
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test_array, y_pred)

recall = recall_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[1897191    1469]
 [  23004  242169]]

Attack-class metrics
--------------------
Accuracy : 0.9887 (98.87%)
Detection rate : 0.9132 (91.32%)
Precision      : 0.9940 (99.40%)
FPR            : 0.0008 (0.08%)
F1 score       : 0.9519 (95.19%)


In [19]:
rnn_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/Models_Table_7/RNN_Wang.keras"
)

In [20]:
current_test_input = X_test[:, np.newaxis, :]
current_probabilities = rnn_model.predict(
        current_test_input,
        batch_size=2048,
        verbose=1,
    ).reshape(-1)

y_pred = (
    current_probabilities >= 0.5
).astype("int8")

cm = confusion_matrix(
    y_test_array,
    y_pred,
    labels=[0, 1],
)


1057/1057 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step


In [21]:
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test_array, y_pred)

recall = recall_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test_array,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[1891879    6781]
 [  37066  228107]]

Attack-class metrics
--------------------
Accuracy : 0.9797 (97.97%)
Detection rate : 0.8602 (86.02%)
Precision      : 0.9711 (97.11%)
FPR            : 0.0036 (0.36%)
F1 score       : 0.9123 (91.23%)


# **Lightweight 1D CNN for binary CSE-CIC-IDS2018 intrusion detection**

In [2]:
dataframe = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/cicids2018_preprocessed.parquet")

In [3]:
random.seed(42)
np.random.seed(42)
tf.keras.utils.set_random_seed(42)

In [4]:
dataframe.columns = dataframe.columns.astype(str).str.strip()

column_aliases = {
    "Source IP": "Src IP",
    "Source Port": "Src Port",
    "Destination IP": "Dst IP",
    "Destination Port": "Dst Port",
    "Flow Bytes/s": "Flow Byts/s",
    "Flow Packets/s": "Flow Pkts/s",
}

dataframe = dataframe.rename(
    columns={
        column: column_aliases[column]
        for column in dataframe.columns
        if column in column_aliases
    }
)

label_column = "Label"
dataframe[label_column] = dataframe[label_column].astype(str).str.strip()

dataframe = dataframe.loc[
    ~dataframe[label_column].str.casefold().isin(
        ["", "nan", "none", "label"]
    )
].reset_index(drop=True)

dataframe["Binary Label"] = (
    dataframe[label_column].str.casefold() != "benign"
).astype("int8")

print(dataframe["Binary Label"].value_counts())
print(dataframe["Binary Label"].value_counts(normalize=True))

Binary Label
0    13353852
1     2354692
Name: count, dtype: int64
Binary Label
0    0.850101
1    0.149899
Name: proportion, dtype: float64


In [5]:
import gc

print("Before duplicate removal:", dataframe.shape)
dataframe = dataframe.drop_duplicates().reset_index(drop=True)
print("After duplicate removal:", dataframe.shape)

candidate_feature_columns = [
    column
    for column in dataframe.columns
    if column not in [label_column, "Binary Label"]
]

for column in candidate_feature_columns:
    if dataframe[column].dtype == "object":
        converted_values = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        )

        if converted_values.notna().sum() >= 0.95 * dataframe[column].notna().sum():
            dataframe[column] = converted_values

numeric_columns_now = dataframe[
    candidate_feature_columns
].select_dtypes(include=[np.number]).columns.tolist()

dataframe[numeric_columns_now] = dataframe[
    numeric_columns_now
].replace([np.inf, -np.inf], np.nan)

rows_before_missing_removal = dataframe.shape[0]
dataframe = dataframe.dropna(axis=0).reset_index(drop=True)
rows_after_missing_removal = dataframe.shape[0]

print(
    "Rows removed for NaN/infinity:",
    rows_before_missing_removal - rows_after_missing_removal,
)
print("Cleaned shape:", dataframe.shape)

gc.collect()

Before duplicate removal: (15708544, 80)
After duplicate removal: (11917125, 80)
Rows removed for NaN/infinity: 987
Cleaned shape: (11916138, 80)


0

In [6]:
X_all_raw = dataframe.drop(
    columns=[label_column, "Binary Label"]
)
y_all = dataframe["Binary Label"]

X_train_validation_raw, X_test_raw, y_train_validation, y_test = train_test_split(
    X_all_raw,
    y_all,
    test_size=0.196,
    random_state=42,
    stratify=y_all,
    shuffle=True,
)

validation_fraction_within_train_validation = 0.078 / (1.0 - 0.196)

X_train_raw, X_validation_raw, y_train, y_validation = train_test_split(
    X_train_validation_raw,
    y_train_validation,
    test_size=validation_fraction_within_train_validation,
    random_state=42,
    stratify=y_train_validation,
    shuffle=True,
)

print("Training:", X_train_raw.shape)
print("Validation:", X_validation_raw.shape)
print("Test:", X_test_raw.shape)

print("Observed fractions:")
print("Training:", X_train_raw.shape[0] / dataframe.shape[0])
print("Validation:", X_validation_raw.shape[0] / dataframe.shape[0])
print("Test:", X_test_raw.shape[0] / dataframe.shape[0])

Training: (8651115, 78)
Validation: (929459, 78)
Test: (2335564, 78)
Observed fractions:
Training: 0.7259999003032694
Validation: 0.07800001980507443
Test: 0.19600007989165616


In [7]:
numeric_columns = X_train_raw.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_columns = [
    column
    for column in X_train_raw.columns
    if column not in numeric_columns
]

retained_categorical_columns = [
    column
    for column in categorical_columns
    if X_train_raw[column].nunique(dropna=False) <= 100
]

removed_high_cardinality_columns = [
    column
    for column in categorical_columns
    if column not in retained_categorical_columns
]

print("Numeric columns:", len(numeric_columns))
print("Retained categorical columns:", retained_categorical_columns)
print("Removed high-cardinality columns:", removed_high_cardinality_columns)

Numeric columns: 78
Retained categorical columns: []
Removed high-cardinality columns: []


In [8]:
constant_numeric_columns = [
    column
    for column in numeric_columns
    if X_train_raw[column].nunique(dropna=False) <= 1
]

retained_numeric_columns = [
    column
    for column in numeric_columns
    if column not in constant_numeric_columns
]

print("Constant numeric columns removed:", constant_numeric_columns)
print("Numeric columns retained:", len(retained_numeric_columns))

Constant numeric columns removed: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg']
Numeric columns retained: 70


In [9]:
training_correlations = X_train_raw[
    retained_numeric_columns
].corr(method="pearson").abs()

upper_triangle_mask = np.triu(
    np.ones(training_correlations.shape),
    k=1,
).astype(bool)

upper_triangle = training_correlations.where(
    upper_triangle_mask
)

correlated_columns_to_remove = [
    column
    for column in upper_triangle.columns
    if (upper_triangle[column] > 0.95).any()
]

retained_numeric_columns = [
    column
    for column in retained_numeric_columns
    if column not in correlated_columns_to_remove
]

print("Correlated columns removed:", len(correlated_columns_to_remove))
print(correlated_columns_to_remove)
print("Numeric columns retained:", len(retained_numeric_columns))

Correlated columns removed: 29
['TotLen Bwd Pkts', 'Fwd Pkt Len Std', 'Bwd Pkt Len Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Pkt Len Max', 'Pkt Len Std', 'SYN Flag Cnt', 'CWE Flag Count', 'ECE Flag Cnt', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Subflow Fwd Pkts', 'Subflow Fwd Byts', 'Subflow Bwd Pkts', 'Subflow Bwd Byts', 'Fwd Act Data Pkts', 'Idle Mean', 'Idle Std', 'Idle Max']
Numeric columns retained: 41


In [10]:
numeric_scaler = MinMaxScaler(feature_range=(0, 1))

X_train_numeric = numeric_scaler.fit_transform(
    X_train_raw[retained_numeric_columns]
).astype("float32")

X_validation_numeric = numeric_scaler.transform(
    X_validation_raw[retained_numeric_columns]
).astype("float32")

X_test_numeric = numeric_scaler.transform(
    X_test_raw[retained_numeric_columns]
).astype("float32")

X_train_numeric = np.clip(X_train_numeric, 0.0, 1.0)
X_validation_numeric = np.clip(X_validation_numeric, 0.0, 1.0)
X_test_numeric = np.clip(X_test_numeric, 0.0, 1.0)

if retained_categorical_columns:
    try:
        categorical_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            dtype=np.float32,
        )
    except TypeError:
        categorical_encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            dtype=np.float32,
        )

    X_train_categorical = categorical_encoder.fit_transform(
        X_train_raw[retained_categorical_columns].astype(str)
    )
    X_validation_categorical = categorical_encoder.transform(
        X_validation_raw[retained_categorical_columns].astype(str)
    )
    X_test_categorical = categorical_encoder.transform(
        X_test_raw[retained_categorical_columns].astype(str)
    )

    X_train_matrix = np.concatenate(
        [X_train_numeric, X_train_categorical],
        axis=1,
    ).astype("float32")

    X_validation_matrix = np.concatenate(
        [X_validation_numeric, X_validation_categorical],
        axis=1,
    ).astype("float32")

    X_test_matrix = np.concatenate(
        [X_test_numeric, X_test_categorical],
        axis=1,
    ).astype("float32")
else:
    categorical_encoder = None
    X_train_matrix = X_train_numeric
    X_validation_matrix = X_validation_numeric
    X_test_matrix = X_test_numeric

print("Training matrix:", X_train_matrix.shape)
print("Validation matrix:", X_validation_matrix.shape)
print("Test matrix:", X_test_matrix.shape)

Training matrix: (8651115, 41)
Validation matrix: (929459, 41)
Test matrix: (2335564, 41)


In [11]:
from sklearn.utils.class_weight import compute_class_weight

class_labels = np.array([0, 1], dtype="int8")

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=class_labels,
    y=y_train.to_numpy(),
)

class_weight_dictionary = {
    int(class_label): float(class_weight_value)
    for class_label, class_weight_value in zip(
        class_labels,
        class_weight_values,
    )
}

print(class_weight_dictionary)

{0: 0.563956261930375, 1: 4.408921385558133}


In [12]:
from imblearn.over_sampling import SMOTE

training_class_counts = np.bincount(
    y_train.to_numpy().astype("int8"),
    minlength=2,
)

minority_to_majority_ratio = (
    training_class_counts.min()
    / training_class_counts.max()
)

apply_smote = (
    minority_to_majority_ratio < 0.25
    and X_train_matrix.shape[0] < 1_000_000
)

print("Training counts:", training_class_counts)
print("Minority/majority ratio:", minority_to_majority_ratio)
print("Apply SMOTE:", apply_smote)

if apply_smote:
    smote = SMOTE(random_state=42)

    X_train_model, y_train_model = smote.fit_resample(
        X_train_matrix,
        y_train.to_numpy().astype("int8"),
    )

    X_train_model = X_train_model.astype("float32")
    y_train_model = y_train_model.astype("float32")
else:
    X_train_model = X_train_matrix
    y_train_model = y_train.to_numpy().astype("float32")

X_validation_model = X_validation_matrix
y_validation_model = y_validation.to_numpy().astype("float32")

X_test_model = X_test_matrix
y_test_model = y_test.to_numpy().astype("int8")

print("Training shape used by model:", X_train_model.shape)
print(
    "Training counts used by model:",
    np.bincount(y_train_model.astype("int8")),
)

Training counts: [7670023  981092]
Minority/majority ratio: 0.12791252386074983
Apply SMOTE: False
Training shape used by model: (8651115, 41)
Training counts used by model: [7670023  981092]


In [13]:
X_train_model = X_train_model[..., np.newaxis]
X_validation_model = X_validation_model[..., np.newaxis]
X_test_model = X_test_model[..., np.newaxis]

feature_count = X_train_model.shape[1]

print("Training input:", X_train_model.shape)
print("Validation input:", X_validation_model.shape)
print("Test input:", X_test_model.shape)

Training input: (8651115, 41, 1)
Validation input: (929459, 41, 1)
Test input: (2335564, 41, 1)


In [14]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(shape=(feature_count, 1)),
        tf.keras.layers.Conv1D(
            filters=32,
            kernel_size=3,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Conv1D(
            filters=16,
            kernel_size=3,
            padding="same",
            activation="relu",
        ),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dropout(0.30),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ],
    name="Lightweight_1D_CNN_CSE_CIC_IDS2018",
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc"),
    ],
)

model.summary()
print("Total parameters:", model.count_params())

Model: "Lightweight_1D_CNN_CSE_CIC_IDS2018"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 41, 32)         │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 41, 32)         │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 41, 16)         │         1,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 41, 16)         │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,889 (7.38 KB)

 Trainable params: 1,793 (7.00 KB)

 Non-trainable params: 96 (384.00 B)

Total parameters: 1889


In [15]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=4,
    restore_best_weights=True,
)

reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-5,
)

history = model.fit(
    X_train_model,
    y_train_model,
    validation_data=(
        X_validation_model,
        y_validation_model,
    ),
    batch_size=2048,
    epochs=20,
    shuffle=True,
    class_weight=class_weight_dictionary,
    callbacks=[
        early_stopping,
        reduce_learning_rate,
    ],
    verbose=1,
)

Epoch 1/20
4225/4225 ━━━━━━━━━━━━━━━━━━━━ 22s 4ms/step - accuracy: 0.9294 - auc: 0.9541 - loss: 0.2541 - val_accuracy: 0.9673 - val_auc: 0.9640 - val_loss: 0.1093 - learning_rate: 0.0010
Epoch 2/20
4225/4225 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.9619 - auc: 0.9645 - loss: 0.2017 - val_accuracy: 0.9575 - val_auc: 0.9699 - val_loss: 0.1805 - learning_rate: 0.0010
Epoch 3/20
4225/4225 ━━━━━━━━━━━━━━━━━━━━ 13s 3ms/step - accuracy: 0.9682 - auc: 0.9671 - loss: 0.1902 - val_accuracy: 0.9764 - val_auc: 0.9708 - val_loss: 0.1093 - learning_rate: 0.0010
Epoch 4/20
4225/4225 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.9716 - auc: 0.9683 - loss: 0.1833 - val_accuracy: 0.9839 - val_auc: 0.9719 - val_loss: 0.0774 - learning_rate: 0.0010
Epoch 5/20
4225/4225 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.9739 - auc: 0.9689 - loss: 0.1793 - val_accuracy: 0.9843 - val_auc: 0.9713 - val_loss: 0.0779 - learning_rate: 0.0010
Epoch 6/20
4225/4225 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy

In [16]:
validation_probabilities = model.predict(
    X_validation_model,
    batch_size=2048,
    verbose=1,
).reshape(-1)

validation_truth = y_validation_model.astype("int8")

print(validation_probabilities.shape)

454/454 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
(929459,)


In [18]:
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
)
candidate_thresholds = np.round(
    np.arange(0.05, 0.951, 0.01),
    2,
)

threshold_rows = []

for candidate_threshold in candidate_thresholds:
    validation_predictions = (
        validation_probabilities >= candidate_threshold
    ).astype("int8")

    threshold_rows.append(
        {
            "Threshold": candidate_threshold,
            "Balanced accuracy": balanced_accuracy_score(
                validation_truth,
                validation_predictions,
            ),
            "Precision": precision_score(
                validation_truth,
                validation_predictions,
                pos_label=1,
                zero_division=0,
            ),
            "Recall": recall_score(
                validation_truth,
                validation_predictions,
                pos_label=1,
                zero_division=0,
            ),
            "F1": f1_score(
                validation_truth,
                validation_predictions,
                pos_label=1,
                zero_division=0,
            ),
        }
    )

threshold_results = pd.DataFrame(threshold_rows)

best_threshold_row = threshold_results.loc[
    threshold_results["Balanced accuracy"].idxmax()
]

optimal_threshold = float(
    best_threshold_row["Threshold"]
)

print("Optimal threshold:", optimal_threshold)
print(best_threshold_row)

Optimal threshold: 0.28
Threshold            0.280000
Balanced accuracy    0.949313
Precision            0.855969
Recall               0.918393
F1                   0.886083
Name: 23, dtype: float64


In [19]:
test_probabilities = model.predict(
    X_test_model,
    batch_size=2048,
    verbose=1,
).reshape(-1)

y_pred = (
    test_probabilities >= optimal_threshold
).astype("int8")

1141/1141 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [20]:
cm = confusion_matrix(
    y_test_model,
    y_pred,
    labels=[0, 1],
)
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test_model, y_pred)

recall = recall_score(
    y_test_model,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test_model,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test_model,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[2029944   40752]
 [  21534  243334]]

Attack-class metrics
--------------------
Accuracy : 0.9733 (97.33%)
Detection rate : 0.9187 (91.87%)
Precision      : 0.8566 (85.66%)
FPR            : 0.0197 (1.97%)
F1 score       : 0.8865 (88.65%)


In [21]:
model.save("/content/drive/MyDrive/Models_Table_7/1DCNN_Anwar.keras")

# **Ensemble-based binary intrusion detection on CSE-CIC-IDS2018**

In [2]:
dataframe = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/DATASETS/CIC-IDS-2018/cicids2018_preprocessed.parquet")

In [3]:
random.seed(42)
np.random.seed(42)

In [4]:
dataframe.columns = dataframe.columns.astype(str).str.strip()

column_aliases = {
    "Source IP": "Src IP",
    "Source Port": "Src Port",
    "Destination IP": "Dst IP",
    "Destination Port": "Dst Port",
    "Flow Bytes/s": "Flow Byts/s",
    "Flow Packets/s": "Flow Pkts/s",
}

dataframe = dataframe.rename(
    columns={
        column: column_aliases[column]
        for column in dataframe.columns
        if column in column_aliases
    }
)

print("Columns:", len(dataframe.columns))

Columns: 80


In [5]:
import gc

label_column = "Label"
dataframe[label_column] = dataframe[label_column].astype(str).str.strip()

dataframe = dataframe.loc[
    ~dataframe[label_column].str.casefold().isin(
        ["", "nan", "none", "label"]
    )
].reset_index(drop=True)

print("Before duplicate removal:", dataframe.shape)
dataframe = dataframe.drop_duplicates().reset_index(drop=True)
print("After duplicate removal:", dataframe.shape)

candidate_columns = [
    column
    for column in dataframe.columns
    if column != label_column
]

for column in candidate_columns:
    if dataframe[column].dtype == "object":
        numeric_conversion = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        )

        if numeric_conversion.notna().sum() >= 0.95 * dataframe[column].notna().sum():
            dataframe[column] = numeric_conversion

numeric_columns = dataframe[
    candidate_columns
].select_dtypes(include=[np.number]).columns.tolist()

dataframe[numeric_columns] = dataframe[
    numeric_columns
].replace([np.inf, -np.inf], np.nan)

rows_before_dropna = dataframe.shape[0]
dataframe = dataframe.dropna(axis=0).reset_index(drop=True)
rows_after_dropna = dataframe.shape[0]

print("Rows removed for missing/invalid values:", rows_before_dropna - rows_after_dropna)
print("Cleaned shape:", dataframe.shape)

gc.collect()

Before duplicate removal: (15708544, 80)
After duplicate removal: (11917125, 80)
Rows removed for missing/invalid values: 987
Cleaned shape: (11916138, 80)


0

In [6]:
dataframe["Binary Label"] = (
    dataframe[label_column].str.casefold() != "benign"
).astype("int8")

target_sample_size = min(1_048_575, dataframe.shape[0])

if target_sample_size < dataframe.shape[0]:
    sampled_indices, unused_indices = train_test_split(
        np.arange(dataframe.shape[0]),
        train_size=target_sample_size,
        random_state=42,
        stratify=dataframe["Binary Label"],
        shuffle=True,
    )

    dataframe = dataframe.iloc[
        sampled_indices
    ].reset_index(drop=True)

    del unused_indices
    del sampled_indices
    gc.collect()

print("Experimental sample shape:", dataframe.shape)
print(dataframe["Binary Label"].value_counts())
print(dataframe["Binary Label"].value_counts(normalize=True))

Experimental sample shape: (1048575, 80)
Binary Label
0    929660
1    118915
Name: count, dtype: int64
Binary Label
0    0.886594
1    0.113406
Name: proportion, dtype: float64


In [7]:
identifier_columns = [
    "Flow ID",
    "Src IP",
    "Dst IP",
    "Timestamp",
]

X_raw = dataframe.drop(
    columns=[
        label_column,
        "Binary Label",
        *[
            column
            for column in identifier_columns
            if column in dataframe.columns
        ],
    ],
    errors="ignore",
)

y = dataframe["Binary Label"].copy()

print("Raw predictors after identifier removal:", X_raw.shape[1])
print(X_raw.dtypes.value_counts())

Raw predictors after identifier removal: 78
float64    78
Name: count, dtype: int64


In [8]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
    shuffle=True,
)

print("Training:", X_train_raw.shape)
print("Testing:", X_test_raw.shape)
print("Training classes:", y_train.value_counts().to_dict())
print("Testing classes:", y_test.value_counts().to_dict())

Training: (838860, 78)
Testing: (209715, 78)
Training classes: {0: 743728, 1: 95132}
Testing classes: {0: 185932, 1: 23783}


In [9]:
numeric_feature_columns = X_train_raw.select_dtypes(
    include=[np.number]
).columns.tolist()

categorical_feature_columns = [
    column
    for column in X_train_raw.columns
    if column not in numeric_feature_columns
]

print("Numeric features:", len(numeric_feature_columns))
print("Categorical features:", categorical_feature_columns)

Numeric features: 78
Categorical features: []


In [10]:
standard_scaler = StandardScaler()

X_train_numeric = standard_scaler.fit_transform(
    X_train_raw[numeric_feature_columns]
).astype("float32")

X_test_numeric = standard_scaler.transform(
    X_test_raw[numeric_feature_columns]
).astype("float32")

X_train_processed = pd.DataFrame(
    X_train_numeric,
    columns=numeric_feature_columns,
    index=X_train_raw.index,
)

X_test_processed = pd.DataFrame(
    X_test_numeric,
    columns=numeric_feature_columns,
    index=X_test_raw.index,
)

categorical_encoders = {}

for column in categorical_feature_columns:
    label_encoder = LabelEncoder()

    training_values = X_train_raw[column].astype(str)
    testing_values = X_test_raw[column].astype(str)

    label_encoder.fit(training_values)
    categorical_encoders[column] = label_encoder

    training_mapping = {
        category: index
        for index, category in enumerate(label_encoder.classes_)
    }

    X_train_processed[column] = training_values.map(
        training_mapping
    ).astype("float32")

    X_test_processed[column] = testing_values.map(
        training_mapping
    ).fillna(-1).astype("float32")

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (838860, 78)
Processed testing shape: (209715, 78)


In [11]:
training_with_target = X_train_processed.copy()
training_with_target["Binary Label"] = y_train.to_numpy()

target_correlations = training_with_target.corr(
    numeric_only=True
)["Binary Label"].drop("Binary Label").abs()

correlation_selected_features = target_correlations.loc[
    target_correlations > 0.5
].index.tolist()

print("Correlation-selected features:", len(correlation_selected_features))
print(correlation_selected_features)

Correlation-selected features: 0
[]


In [12]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

mutual_information_k = min(
    20,
    X_train_processed.shape[1],
)

mutual_information_selector = SelectKBest(
    score_func=mutual_info_classif,
    k=mutual_information_k,
)

mutual_information_selector.fit(
    X_train_processed,
    y_train,
)

mutual_information_selected_features = X_train_processed.columns[
    mutual_information_selector.get_support()
].tolist()

print("Mutual-information features:", len(mutual_information_selected_features))
print(mutual_information_selected_features)

Mutual-information features: 20
['Dst Port', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Max', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'Pkt Size Avg', 'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Subflow Fwd Byts', 'Subflow Bwd Byts', 'Init Fwd Win Byts', 'Init Bwd Win Byts']


In [13]:
from sklearn.decomposition import PCA

pca_model = PCA(
    n_components=0.95,
    svd_solver="full",
)

pca_model.fit(X_train_processed)

pca_dominant_feature_indices = np.unique(
    np.argmax(
        np.abs(pca_model.components_),
        axis=1,
    )
)

pca_selected_features = X_train_processed.columns[
    pca_dominant_feature_indices
].tolist()

print("PCA components:", pca_model.n_components_)
print("PCA-mapped original features:", len(pca_selected_features))
print(pca_selected_features)

PCA components: 24
PCA-mapped original features: 20
['Tot Bwd Pkts', 'Fwd Pkt Len Mean', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Fwd URG Flags', 'Fwd Header Len', 'Pkt Len Std', 'Pkt Len Var', 'FIN Flag Cnt', 'ECE Flag Cnt', 'Down/Up Ratio', 'Subflow Bwd Byts', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Active Std', 'Active Max', 'Idle Std']


In [14]:
final_selected_features = sorted(
    set(correlation_selected_features)
    .union(mutual_information_selected_features)
    .union(pca_selected_features)
)

if not final_selected_features:
    raise ValueError("Feature-selection union is empty.")

X_train_selected = X_train_processed[
    final_selected_features
].copy()

X_test_selected = X_test_processed[
    final_selected_features
].copy()

print("Final selected feature count:", len(final_selected_features))
print(final_selected_features)
print("Training shape:", X_train_selected.shape)
print("Testing shape:", X_test_selected.shape)

Final selected feature count: 34
['Active Max', 'Active Std', 'Bwd IAT Min', 'Bwd Pkt Len Max', 'Bwd Pkt Len Mean', 'Bwd Pkt Len Std', 'Bwd Seg Size Avg', 'Down/Up Ratio', 'Dst Port', 'ECE Flag Cnt', 'FIN Flag Cnt', 'Flow Byts/s', 'Flow IAT Max', 'Flow Pkts/s', 'Fwd Header Len', 'Fwd PSH Flags', 'Fwd Pkt Len Max', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Fwd Seg Size Avg', 'Fwd URG Flags', 'Idle Std', 'Init Bwd Win Byts', 'Init Fwd Win Byts', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var', 'Pkt Size Avg', 'Subflow Bwd Byts', 'Subflow Fwd Byts', 'Tot Bwd Pkts', 'TotLen Bwd Pkts', 'TotLen Fwd Pkts']
Training shape: (838860, 34)
Testing shape: (209715, 34)


In [15]:
random_forest_model = RandomForestClassifier(
    n_estimators=10,
    max_depth=None,
    criterion="gini",
    min_samples_split=2,
    min_samples_leaf=1,
    max_leaf_nodes=None,
    min_weight_fraction_leaf=0.0,
    oob_score=False,
    max_features="sqrt",
    min_impurity_decrease=0.0,
    bootstrap=True,
    n_jobs=None,
    random_state=42,
    verbose=0,
    class_weight=None,
    warm_start=False,
    ccp_alpha=0.0,
    max_samples=None,
)



In [16]:
random_forest_model.fit(
    X_train_selected,
    y_train,
)

print("Random Forest trained.")

Random Forest trained.


In [17]:
y_pred = random_forest_model.predict(
        X_test_selected
    )


cm = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1],
    )

In [18]:
tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_test, y_pred)

recall = recall_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)

precision = precision_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)



false_positive_rate = (
    fp / (fp + tn)
    if (fp + tn) > 0
    else 0.0
)

f1 = f1_score(
    y_test,
    y_pred,
    pos_label=1,
    zero_division=0,
)


print("Confusion matrix")
print("----------------")
print(cm)

print("\nAttack-class metrics")
print("--------------------")
print(f"Accuracy : {accuracy:.4f} "
      f"({accuracy * 100:.2f}%)")
print(f"Detection rate : {recall:.4f} "
      f"({recall * 100:.2f}%)")
print(f"Precision      : {precision:.4f} "
      f"({precision * 100:.2f}%)")
print(f"FPR            : {false_positive_rate:.4f} "
      f"({false_positive_rate * 100:.2f}%)")
print(f"F1 score       : {f1:.4f} "
      f"({f1 * 100:.2f}%)")

Confusion matrix
----------------
[[185190    742]
 [  2300  21483]]

Attack-class metrics
--------------------
Accuracy : 0.9855 (98.55%)
Detection rate : 0.9033 (90.33%)
Precision      : 0.9666 (96.66%)
FPR            : 0.0040 (0.40%)
F1 score       : 0.9339 (93.39%)


# New Section